# 02 · Train the U-Net (Sentinel-1 only)

- **Model:** U-Net, ResNet-34 encoder pretrained on ImageNet (`segmentation_models_pytorch`), 3 input channels (VV, VH, VV − VH in dB, standardised)
- **Loss:** class-weighted cross-entropy + 0.5 × Dice (handles imbalance: built-up, water and bare are rare)
- **Sampling:** random 256 × 256 crops from random region-months, with rotations and flips
- **Selection:** checkpoint with the best validation mIoU
- **Hardware:** a GPU is strongly recommended (Colab or Kaggle T4 is enough). CPU works only for a smoke test.

In [1]:
# --- Colab / Kaggle only: uncomment and adapt ---
# from google.colab import drive; drive.mount('/content/drive')
# !pip -q install segmentation-models-pytorch pyproj
# %cd /content/drive/MyDrive/s1-lulc        # folder containing config.py and data/

In [2]:
import sys, json, time, random
from pathlib import Path

ROOT = Path.cwd() if Path("config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

import config as C
from train_utils import (load_regions, class_counts, class_weights_from_counts,
                         RandomCropDataset, TiledDataset, seed_worker,
                         CombinedLoss, train_one_epoch, evaluate)
from lulc_model import build_model, save_checkpoint

DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"; MODEL_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

Device: cuda NVIDIA H200 NVL


## 1 · Hyperparameters

In [3]:
CFG = dict(
    encoder="resnet34",
    epochs=60,
    batch_size=64,
    samples_per_epoch=12000,
    lr=6e-4,
    weight_decay=1e-4,
    dice_weight=0.5,
    num_workers=0,
)
SMOKE_TEST = DEVICE == "cpu"
if SMOKE_TEST:
    CFG.update(epochs=2, samples_per_epoch=32, batch_size=4, num_workers=0)
    print("CPU detected -> smoke-test settings. Use a GPU for the real run.")
CFG

{'encoder': 'resnet34',
 'epochs': 60,
 'batch_size': 64,
 'samples_per_epoch': 12000,
 'lr': 0.0006,
 'weight_decay': 0.0001,
 'dice_weight': 0.5,
 'num_workers': 0}

## 2 · Data

In [4]:
train_regions = load_regions(DATA_DIR, "train")
val_regions = load_regions(DATA_DIR, "val")
stats = json.loads((DATA_DIR / "norm_stats.json").read_text())
MEAN, STD = stats["mean"], stats["std"]

counts = class_counts(train_regions)
class_w = class_weights_from_counts(counts)
pd.DataFrame({"pixels": counts, "share_%": 100 * counts / counts.sum(), "loss_weight": class_w.numpy()},
             index=C.CLASS_NAMES).round(3)

,pixels,share_%,loss_weight
Tree cover,6648132,28.178,0.539
Shrub & grassland,4207275,17.833,0.678
Cropland,3298905,13.983,0.765
Built-up,6515542,27.616,0.545
Bare / sparse,282155,1.196,2.617
Water,2640951,11.194,0.856


In [5]:
train_ds = RandomCropDataset(train_regions, MEAN, STD, samples_per_epoch=CFG["samples_per_epoch"], seed=SEED)
val_ds = TiledDataset(val_regions, MEAN, STD)
if SMOKE_TEST:
    val_ds.items = val_ds.items[:16]

train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=False,
                      num_workers=CFG["num_workers"], worker_init_fn=seed_worker,
                      pin_memory=DEVICE == "cuda", drop_last=True)
val_dl = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                    num_workers=CFG["num_workers"], pin_memory=DEVICE == "cuda")
print(f"train: {len(train_regions)} regions, {len(train_ds.index)} usable region-months, "
      f"{len(train_ds)} crops per epoch")
print(f"val:   {len(val_regions)} regions, {len(val_ds)} tiles")

train: 40 regions, 480 usable region-months, 12000 crops per epoch
val:   7 regions, 756 tiles


## 3 · Model, loss, optimiser

In [6]:
model = build_model(CFG["encoder"], "imagenet").to(DEVICE)
loss_fn = CombinedLoss(class_w.to(DEVICE), CFG["dice_weight"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
steps = CFG["epochs"] * len(train_dl)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG["lr"], total_steps=steps, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

24.4 M parameters


## 4 · Training loop

In [7]:
history = []
best_miou, best_epoch = -1.0, -1
t0 = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    tr = train_one_epoch(model, train_dl, loss_fn, optimizer, scheduler, scaler, DEVICE)
    va = evaluate(model, val_dl, loss_fn, DEVICE)
    history.append({"epoch": epoch,
                    "train_loss": tr["loss"], "val_loss": va["loss"],
                    "train_mIoU": tr["mIoU"], "val_mIoU": va["mIoU"],
                    "train_OA": tr["overall_accuracy"], "val_OA": va["overall_accuracy"],
                    **{f"val_IoU_{n}": v for n, v in zip(C.CLASS_NAMES, va["IoU"])},
                    "lr": optimizer.param_groups[0]["lr"]})
    flag = ""
    if va["mIoU"] > best_miou:
        best_miou, best_epoch = va["mIoU"], epoch
        save_checkpoint(MODEL_DIR / "best_model.pth", model, CFG["encoder"], MEAN, STD,
                        extra={"epoch": epoch, "val_mIoU": best_miou, "cfg": CFG})
        flag = "  * saved"
    print(f"ep {epoch:3d} | loss {tr['loss']:.3f} / {va['loss']:.3f} | "
          f"mIoU {tr['mIoU']:.3f} / {va['mIoU']:.3f} | OA {va['overall_accuracy']:.3f} | "
          f"{(time.time() - t0) / 60:.1f} min{flag}")

hist = pd.DataFrame(history)
hist.to_csv(MODEL_DIR / "training_history.csv", index=False)
print(f"Best val mIoU {best_miou:.3f} at epoch {best_epoch}")

ep   1 | loss 1.481 / 1.222 | mIoU 0.402 / 0.563 | OA 0.709 | 2.7 min  * saved


ep   2 | loss 0.856 / 0.961 | mIoU 0.637 / 0.627 | OA 0.773 | 5.4 min  * saved


ep   3 | loss 0.655 / 0.911 | mIoU 0.666 / 0.632 | OA 0.763 | 7.8 min  * saved


ep   4 | loss 0.630 / 0.829 | mIoU 0.661 / 0.643 | OA 0.770 | 10.3 min  * saved


ep   5 | loss 0.554 / 0.835 | mIoU 0.687 / 0.649 | OA 0.785 | 13.0 min  * saved


ep   6 | loss 0.579 / 1.178 | mIoU 0.676 / 0.557 | OA 0.688 | 15.6 min


... (36 outputs left out)

ep  43 | loss 0.364 / 1.082 | mIoU 0.772 / 0.618 | OA 0.791 | 108.5 min


ep  44 | loss 0.362 / 1.123 | mIoU 0.772 / 0.617 | OA 0.788 | 112.0 min


ep  45 | loss 0.355 / 1.134 | mIoU 0.776 / 0.618 | OA 0.787 | 115.3 min


ep  46 | loss 0.358 / 1.237 | mIoU 0.773 / 0.604 | OA 0.782 | 118.5 min


## 5 · Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist.epoch, hist.train_loss, label="train"); axes[0].plot(hist.epoch, hist.val_loss, label="val")
axes[0].set_title("Loss (CE + Dice)")
axes[1].plot(hist.epoch, hist.train_mIoU, label="train"); axes[1].plot(hist.epoch, hist.val_mIoU, label="val")
axes[1].axvline(best_epoch, ls="--", c="grey", lw=1, label=f"best (ep {best_epoch})")
axes[1].set_title("mIoU")
axes[2].plot(hist.epoch, hist.train_OA, label="train"); axes[2].plot(hist.epoch, hist.val_OA, label="val")
axes[2].set_title("Overall accuracy")
for a in axes: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(MODEL_DIR / "training_curves.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for n, col in zip(C.CLASS_NAMES, C.CLASS_COLORS):
    ax.plot(hist.epoch, hist[f"val_IoU_{n}"], label=n, color=col, lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("IoU"); ax.set_title("Validation IoU per class")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(MODEL_DIR / "val_iou_per_class.png", dpi=150); plt.show()

In [ ]:
ax = hist.plot(x="epoch", y="lr", figsize=(6, 3), legend=False, title="Learning rate (OneCycle)")
ax.grid(alpha=.3); plt.show()